### Primary and Secondary Analysis

In [ ]:
import pandas as pd

In [ ]:
city_target_passenger_rating_df = pd.read_csv('city_target_passenger_rating.csv')
city_target_passenger_rating_df.head()

,city_id,target_avg_passenger_rating
0,CH01,8.00
1,UP01,7.25
2,AP01,8.50
3,MP01,8.00
4,RJ01,8.25


In [ ]:
dim_city_df = pd.read_csv('dim_city.csv')
dim_city_df.head()

,city_id,city_name
0,RJ01,Jaipur
1,UP01,Lucknow
2,GJ01,Surat
3,KL01,Kochi
4,MP01,Indore


In [ ]:
dim_date_df = pd.read_csv('dim_date.csv')
dim_date_df.head()

,date,start_of_month,month_name,day_type
0,2024-01-01,2024-01-01,January,Weekday
1,2024-01-02,2024-01-01,January,Weekday
2,2024-01-03,2024-01-01,January,Weekday
3,2024-01-04,2024-01-01,January,Weekday
4,2024-01-05,2024-01-01,January,Weekday


In [ ]:
dim_repeat_trip_distribution_df = pd.read_csv('dim_repeat_trip_distribution.csv')
dim_repeat_trip_distribution_df.head()

,month,city_id,trip_count,repeat_passenger_count
0,2024-01-01,AP01,10-Trips,7
1,2024-01-01,AP01,2-Trips,352
2,2024-01-01,AP01,3-Trips,158
3,2024-01-01,AP01,4-Trips,53
4,2024-01-01,AP01,5-Trips,38


In [ ]:
fact_passenger_summary_df = pd.read_csv('fact_passenger_summary.csv')
fact_passenger_summary_df.head(10)

,month,city_id,new_passengers,repeat_passengers,total_passengers
0,2024-01-01,AP01,2513,650,3163
1,2024-01-01,CH01,3920,720,4640
2,2024-01-01,GJ01,2432,1184,3616
3,2024-01-01,GJ02,2089,544,2633
4,2024-01-01,KA01,1957,172,2129
5,2024-01-01,KL01,4865,795,5660
6,2024-01-01,MP01,2843,1033,3876
7,2024-01-01,RJ01,10423,1422,11845
8,2024-01-01,TN01,1822,392,2214
9,2024-01-01,UP01,3465,1431,4896


In [ ]:
fact_trips_df = pd.read_csv('fact_trips.csv')
fact_trips_df.head()

,trip_id,date,city_id,passenger_type,distance_travelled(km),fare_amount,passenger_rating,driver_rating
0,TRPLUC240113d55de2fb,2024-01-13,UP01,repeated,11,158,5,5
1,TRPVAD240129a3b6dba8,2024-01-29,GJ02,repeated,7,74,5,5
2,TRPCOI240107a42430fb,2024-01-07,TN01,repeated,11,155,8,8
3,TRPKOC240325d7601389,2024-03-25,KL01,repeated,36,427,9,10
4,TRPVIS2406027be97166,2024-06-02,AP01,new,17,265,8,8


In [ ]:
monthly_target_new_passengers_df = pd.read_csv('monthly_target_new_passengers.csv')
monthly_target_new_passengers_df.head()

,month,city_id,target_new_passengers
0,2024-05-01,GJ01,1500
1,2024-05-01,GJ02,1500
2,2024-03-01,GJ01,2000
3,2024-05-01,UP01,2000
4,2024-05-01,MP01,2000


In [ ]:
monthly_target_trips_df = pd.read_csv('monthly_target_trips.csv')
monthly_target_trips_df.head()

,month,city_id,total_target_trips
0,2024-03-01,MP01,7000
1,2024-05-01,KA01,2500
2,2024-04-01,UP01,11000
3,2024-02-01,GJ02,6000
4,2024-05-01,KL01,9000


### 1. Top and Bottom Performing Cities:

*   Indentify the top 3 and bottom 3 cities by total trips over the entire analysis period.


In [ ]:
trips_by_city = fact_trips_df.groupby('city_id')['trip_id'].count().reset_index()

# Calculate total trips per city
city_trip_counts = fact_trips_df.groupby('city_id')['trip_id'].count().reset_index()
city_trip_counts.columns = ['city_id', 'total_trips']

# Top and Bottom 3 cities
top_3 = city_trip_counts.nlargest(3, 'total_trips')
bottom_3 = city_trip_counts.nsmallest(3, 'total_trips')

print("Top 3 Cities:")
print(top_3)

print("\nBottom 3 Cities:")
print(bottom_3)

Top 3 Cities:
  city_id  total_trips
7    RJ01        76888
9    UP01        64299
2    GJ01        54843

Bottom 3 Cities:
  city_id  total_trips
4    KA01        16238
8    TN01        21104
0    AP01        28366


### 2. Average Fare per Trip by City


*   Calculate the average fare per trip for each city and compare it with the city's average trip distance. Identify the cities with the highest and lowest average fare per trip to assess pricing efficiency across locations.



In [ ]:
# Calculate average fare per trip
avg_fare_per_trip = fact_trips_df.groupby('city_id')['fare_amount'].mean().reset_index()
avg_fare_per_trip.columns = ['city_id', 'avg_fare_per_trip']

# Cities with highest and lowest average fare per trip
highest_avg_fare = avg_fare_per_trip.nlargest(3, 'avg_fare_per_trip')
lowest_avg_fare = avg_fare_per_trip.nsmallest(3, 'avg_fare_per_trip')

print("Cities with Highest Average Fare per Trip:")
print(highest_avg_fare)

print("\nCities with Lowest Average Fare per Trip:")
print(lowest_avg_fare)


Cities with Highest Average Fare per Trip:
  city_id  avg_fare_per_trip
7    RJ01         483.918128
5    KL01         335.245079
1    CH01         283.686950

Cities with Lowest Average Fare per Trip:
  city_id  avg_fare_per_trip
2    GJ01         117.272925
3    GJ02         118.566165
9    UP01         147.180376


### 3. Average Ratings by City and Passenger Type


*   Calculate the average passenger and driver ratings for each city, segmented by passenger type (new vs. repeat). Identify cities with the highest and lowest average ratings.



In [ ]:
# Grouping by city_id and passenger_type to calculate average ratings
average_ratings = fact_trips_df.groupby(["city_id", "passenger_type"])[
    ["passenger_rating", "driver_rating"]
].mean().reset_index()

# Cities with the highest and lowest average passenger ratings
highest_passenger_rating_city = average_ratings.loc[
    average_ratings["passenger_rating"].idxmax()
]
lowest_passenger_rating_city = average_ratings.loc[
    average_ratings["passenger_rating"].idxmin()
]

# Cities with the highest and lowest average driver ratings
highest_driver_rating_city = average_ratings.loc[
    average_ratings["driver_rating"].idxmax()
]
lowest_driver_rating_city = average_ratings.loc[
    average_ratings["driver_rating"].idxmin()
]

# Displaying the results
print("Average Ratings by City and Passenger Type:")
print(average_ratings)
print("\nCity with Highest Passenger Rating:")
print(highest_passenger_rating_city)
print("\nCity with Lowest Passenger Rating:")
print(lowest_passenger_rating_city)
print("\nCity with Highest Driver Rating:")
print(highest_driver_rating_city)
print("\nCity with Lowest Driver Rating:")
print(lowest_driver_rating_city)


Average Ratings by City and Passenger Type:
   city_id passenger_type  passenger_rating  driver_rating
0     AP01            new          8.976151       8.979995
1     AP01       repeated          7.989628       8.992701
2     CH01            new          8.489158       7.992120
3     CH01       repeated          7.493798       7.472824
4     GJ01            new          7.984173       6.994925
5     GJ01       repeated          5.995511       6.479441
6     GJ02            new          7.979263       7.004147
7     GJ02       repeated          5.978629       6.481072
8     KA01            new          8.982964       8.982878
9     KA01       repeated          7.978495       8.965767
10    KL01            new          8.987394       8.985350
11    KL01       repeated          8.003665       8.989830
12    MP01            new          8.485837       7.970800
13    MP01       repeated          7.473961       7.477404
14    RJ01            new          8.985018       8.988246
15    RJ01  

### 4. Peak and Low Demand Months by City


*   For each city, identify the month with the highest total trips (peak demand) and the month with the lowest total trips (low demand). This analysis will help Goodcabs understand seasonal patterns and adjust recources accordingly.



In [ ]:
# Extract the month from the date column
fact_trips_df["month"] = pd.to_datetime(fact_trips_df["date"]).dt.to_period("M")

# Group by city and month to calculate total trips
monthly_trip_counts = fact_trips_df.groupby(["city_id", "month"]).size().reset_index(name="total_trips")

# Identify the peak and low-demand months for each city
peak_demand_months = monthly_trip_counts.loc[monthly_trip_counts.groupby("city_id")["total_trips"].idxmax()]
low_demand_months = monthly_trip_counts.loc[monthly_trip_counts.groupby("city_id")["total_trips"].idxmin()]

# Display the results
print("Peak Demand Months by City:")
print(peak_demand_months)
print("\nLow Demand Months by City:")
print(low_demand_months)


Peak Demand Months by City:
   city_id    month  total_trips
3     AP01  2024-04         4938
7     CH01  2024-02         7387
15    GJ01  2024-04         9831
21    GJ02  2024-04         5941
28    KA01  2024-05         3007
34    KL01  2024-05        10014
40    MP01  2024-05         7787
43    RJ01  2024-02        15872
50    TN01  2024-03         3680
55    UP01  2024-02        12060

Low Demand Months by City:
   city_id    month  total_trips
0     AP01  2024-01         4468
9     CH01  2024-04         5566
12    GJ01  2024-01         8358
23    GJ02  2024-06         4685
24    KA01  2024-01         2485
35    KL01  2024-06         6399
41    MP01  2024-06         6288
47    RJ01  2024-06         9842
53    TN01  2024-06         3158
58    UP01  2024-05         9705


### 5. Weekend vs. Weekday Trip Demand by City


*   Compare the total trips taken on weekdays versus weekends for each city over the six-month period. Identify cities with a strong preference for either weekend or weekday trips to understand demand variations.



In [ ]:
# Merge fact_trips with dim_date to get day_type (weekday/weekend)
fact_trips = pd.merge(fact_trips_df, dim_date_df[["date", "day_type"]], on="date", how="left")

# Group by city_id and day_type to calculate total trips
trip_demand_by_day_type = fact_trips.groupby(["city_id", "day_type"]).size().reset_index(name="total_trips")

# Pivot the table to have separate columns for weekday and weekend trips
trip_demand_pivot = trip_demand_by_day_type.pivot(index="city_id", columns="day_type", values="total_trips").reset_index()

# Rename columns for clarity (ensure day_type values are "weekday" and "weekend")
trip_demand_pivot.columns = ["city_id", "weekday_trips", "weekend_trips"]

# Fill NaN values with 0 in case some cities have no trips for a day type
trip_demand_pivot.fillna(0, inplace=True)

# Calculate the preference (difference between weekday and weekend trips)
trip_demand_pivot["preference"] = trip_demand_pivot["weekday_trips"] - trip_demand_pivot["weekend_trips"]

# Identify cities with strong preferences
strong_weekday_cities = trip_demand_pivot[trip_demand_pivot["preference"] > 0]
strong_weekend_cities = trip_demand_pivot[trip_demand_pivot["preference"] < 0]

# Display results
print("Trip Demand by Day Type (Weekday vs. Weekend):")
print(trip_demand_pivot)
print("\nCities with Strong Preference for Weekday Trips:")
print(strong_weekday_cities)
print("\nCities with Strong Preference for Weekend Trips:")
print(strong_weekend_cities)


Trip Demand by Day Type (Weekday vs. Weekend):
  city_id  weekday_trips  weekend_trips  preference
0    AP01          15100          13266        1834
1    CH01          19914          19067         847
2    GJ01          37793          17050       20743
3    GJ02          20310          11716        8594
4    KA01           6424           9814       -3390
5    KL01          22915          27787       -4872
6    MP01          21198          21258         -60
7    RJ01          32491          44397      -11906
8    TN01          12576           8528        4048
9    UP01          49617          14682       34935

Cities with Strong Preference for Weekday Trips:
  city_id  weekday_trips  weekend_trips  preference
0    AP01          15100          13266        1834
1    CH01          19914          19067         847
2    GJ01          37793          17050       20743
3    GJ02          20310          11716        8594
8    TN01          12576           8528        4048
9    UP01          

### 6. Repeat Passenger Frequency and City Contribution Analysis


*   Analyse the frequency of trips taken by repeat passengers in each city (e.g., % of repeat passengers taking 2 trips, 3 trips, etc.). Indentify which cities contribute most to higher trip frequencies among repeat passengers, and examine if there are distinguishable patterns between tourism-focused and business-focused cities.



In [ ]:
# Calculate total repeat passengers by city
city_total_repeat_passengers = (
    dim_repeat_trip_distribution_df.groupby("city_id")["repeat_passenger_count"].sum().reset_index()
)
city_total_repeat_passengers.rename(columns={"repeat_passenger_count": "total_repeat_passengers"}, inplace=True)

# Merge total repeat passengers back with the original data
repeat_trip_distribution = pd.merge(
    dim_repeat_trip_distribution_df,
    city_total_repeat_passengers,
    on="city_id",
    how="left"
)

# Calculate percentage contribution for each trip frequency
repeat_trip_distribution["percentage_contribution"] = (
    repeat_trip_distribution["repeat_passenger_count"] / repeat_trip_distribution["total_repeat_passengers"] * 100
)

# Identify cities contributing most to higher trip frequencies
# Assuming "higher trip frequencies" means repeat passengers with 5+ trips
high_trip_frequencies = repeat_trip_distribution[
    repeat_trip_distribution["trip_count"].apply(lambda x: int(x.split("-")[0]) >= 5)
]
city_high_trip_contributions = (
    high_trip_frequencies.groupby("city_id")["repeat_passenger_count"].sum().reset_index()
)
city_high_trip_contributions.rename(columns={"repeat_passenger_count": "high_trip_contributions"}, inplace=True)

# Merge city high trip contributions back with total passengers for comparison
city_analysis = pd.merge(
    city_total_repeat_passengers,
    city_high_trip_contributions,
    on="city_id",
    how="left"
)

# Fill NaN values in high_trip_contributions with 0
city_analysis["high_trip_contributions"].fillna(0, inplace=True)

# Add a percentage column for high trip contributions
city_analysis["high_trip_percentage"] = (
    city_analysis["high_trip_contributions"] / city_analysis["total_repeat_passengers"] * 100
)

# Sort cities by high trip percentage
city_analysis = city_analysis.sort_values("high_trip_percentage", ascending=False)

# Display results
print("Repeat Passenger Frequency Contribution by City:")
print(repeat_trip_distribution)
print("\nCity Contribution to High Trip Frequencies:")
print(city_analysis)

# For tourism vs. business analysis, you would need a separate city classification table
# You can analyze patterns by merging with that classification and visualizing trends


Repeat Passenger Frequency Contribution by City:
          month city_id trip_count  repeat_passenger_count  \
0    2024-01-01    AP01   10-Trips                       7   
1    2024-01-01    AP01    2-Trips                     352   
2    2024-01-01    AP01    3-Trips                     158   
3    2024-01-01    AP01    4-Trips                      53   
4    2024-01-01    AP01    5-Trips                      38   
..          ...     ...        ...                     ...   
535  2024-06-01    UP01    5-Trips                     272   
536  2024-06-01    UP01    6-Trips                     272   
537  2024-06-01    UP01    7-Trips                     246   
538  2024-06-01    UP01    8-Trips                      83   
539  2024-06-01    UP01    9-Trips                      19   

     total_repeat_passengers  percentage_contribution  
0                       5108                 0.137040  
1                       5108                 6.891151  
2                       5108          

<ipython-input-22-c821f42dd5e9>:39: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  city_analysis["high_trip_contributions"].fillna(0, inplace=True)


### 7. Monthly Target Achievement Analysis for Key Metrics


*   For each city, evaluate monthly performance against targets for total trips, new passengers, and average passenger ratings from targets_db. Determine if each metric met, exceeded, or missed the target, and calculate the percentage difference. Identify any consistent patterns in target achievevement, particularly across tourism versus business - focused cities.



In [ ]:
# Calculate average distance and one-time passenger proportions
city_analysis = fact_trips_df.groupby("city_id").agg(
    avg_distance=("distance_travelled(km)", "mean"),
    total_trips=("trip_id", "count"),
    one_time_passengers=("passenger_type", lambda x: (x == "new").sum())
).reset_index()

# Calculate the proportion of one-time passengers
city_analysis["one_time_passenger_ratio"] = city_analysis["one_time_passengers"] / city_analysis["total_trips"]

# Determine thresholds for classification
distance_threshold = city_analysis["avg_distance"].mean()
one_time_threshold = city_analysis["one_time_passenger_ratio"].mean()

# Classify cities
city_analysis["classification"] = city_analysis.apply(
    lambda row: "tourism" if (row["avg_distance"] > distance_threshold and row["one_time_passenger_ratio"] > one_time_threshold) else "business",
    axis=1
)

# Merge classification back into performance data
performance_data = pd.merge(
    fact_passenger_summary_df,
    monthly_target_trips_df,
    on=["month", "city_id"],
    how="left"
)
performance_data = pd.merge(
    performance_data,
    monthly_target_new_passengers_df,
    on=["month", "city_id"],
    how="left"
)
performance_data = pd.merge(
    performance_data,
    city_target_passenger_rating_df,
    on="city_id",
    how="left"
)
performance_data = pd.merge(
    performance_data,
    city_analysis[["city_id", "classification"]],
    on="city_id",
    how="left"
)

# Calculate metrics and trends
performance_data["trips_status"] = performance_data["total_passengers"] - performance_data["total_target_trips"]
performance_data["trips_percentage_diff"] = (
    (performance_data["total_passengers"] - performance_data["total_target_trips"]) / performance_data["total_target_trips"] * 100
)
performance_data["new_passengers_status"] = performance_data["new_passengers"] - performance_data["target_new_passengers"]
performance_data["new_passengers_percentage_diff"] = (
    (performance_data["new_passengers"] - performance_data["target_new_passengers"]) / performance_data["target_new_passengers"] * 100
)
performance_data["rating_status"] = performance_data["total_passengers"] - performance_data["target_avg_passenger_rating"]
performance_data["rating_percentage_diff"] = (
    (performance_data["total_passengers"] - performance_data["target_avg_passenger_rating"]) / performance_data["target_avg_passenger_rating"] * 100
)

# Analyze performance by classification
classification_analysis = performance_data.groupby("classification")[[
    "trips_percentage_diff",
    "new_passengers_percentage_diff",
    "rating_percentage_diff"
]].mean()

# Display results
print("City Classification:")
print(city_analysis)
print("\nMonthly Performance Against Targets:")
print(performance_data)
print("\nPerformance Trends by Classification:")
print(classification_analysis)


City Classification:
  city_id  avg_distance  total_trips  one_time_passengers  \
0    AP01     22.553938        28366                12747   
1    CH01     23.518714        38981                18908   
2    GJ01     10.997247        54843                11626   
3    GJ02     11.517736        32026                10127   
4    KA01     16.496921        16238                11681   
5    KL01     24.065461        50702                26416   
6    MP01     16.502473        42456                14863   
7    RJ01     30.023125        76888                45856   
8    TN01     14.979198        21104                 8514   
9    UP01     12.512963        64299                16260   

   one_time_passenger_ratio classification  
0                  0.449376        tourism  
1                  0.485057        tourism  
2                  0.211987       business  
3                  0.316212       business  
4                  0.719362       business  
5                  0.521005        to

### 8. Highest and Lowest Repeat Passenger Rate (RPR%) by City and Month

Analyse the Repeat Passenger Rate (RPR%) for each city across the
six-month period. Identify the top 2 and bottom 2 cities based on their RPR% to determine which locations have the strongest and weakest rates.

Similarly, analyse the RPR% by month across all cities and identify the months with the highest and lowest repeat passenger rates. This will help to pinpoint any seasonal patterns or months with higher repeat passenger loyalty.

In [ ]:
# Calculate Repeat Passenger Rate (RPR%) for each city and month
fact_passenger_summary_df["RPR%"] = (
    fact_passenger_summary_df["repeat_passengers"] / fact_passenger_summary_df["total_passengers"] * 100
)

# Identify the top 2 and bottom 2 cities by RPR%
top_cities = fact_passenger_summary_df.groupby("city_id")["RPR%"].mean().nlargest(2).reset_index()
bottom_cities = fact_passenger_summary_df.groupby("city_id")["RPR%"].mean().nsmallest(2).reset_index()

# Analyze RPR% by month across all cities
rpr_by_month = fact_passenger_summary_df.groupby("month").agg(
    total_repeat_passengers=("repeat_passengers", "sum"),
    total_passengers=("total_passengers", "sum")
).reset_index()

rpr_by_month["RPR%"] = (
    rpr_by_month["total_repeat_passengers"] / rpr_by_month["total_passengers"] * 100
)

# Identify months with the highest and lowest RPR%
highest_month = rpr_by_month.nlargest(1, "RPR%")
lowest_month = rpr_by_month.nsmallest(1, "RPR%")

# Display results
print("Top 2 Cities by RPR%:")
print(top_cities)

print("\nBottom 2 Cities by RPR%:")
print(bottom_cities)

print("\nRPR% by Month:")
print(rpr_by_month)

print("\nMonth with Highest RPR%:")
print(highest_month)

print("\nMonth with Lowest RPR%:")
print(lowest_month)


Top 2 Cities by RPR%:
  city_id       RPR%
0    GJ01  42.963123
1    UP01  38.131873

Bottom 2 Cities by RPR%:
  city_id       RPR%
0    KA01  11.208195
1    RJ01  18.329207

RPR% by Month:
        month  total_repeat_passengers  total_passengers       RPR%
0  2024-01-01                     8343             44672  18.676128
1  2024-02-01                     9523             45724  20.827137
2  2024-03-01                    10584             41398  25.566452
3  2024-04-01                    11013             37633  29.264210
4  2024-05-01                    12167             36349  33.472723
5  2024-06-01                     9681             32533  29.757477

Month with Highest RPR%:
        month  total_repeat_passengers  total_passengers       RPR%
4  2024-05-01                    12167             36349  33.472723

Month with Lowest RPR%:
        month  total_repeat_passengers  total_passengers       RPR%
0  2024-01-01                     8343             44672  18.676128


### Further analysis and recommendations:

### 1. Factors Influencing Repeat Passenger Rates

What factors (such as quality of service, competitive pricing, or city demographics) might contribute to higher or lower repeat passenger rates in different cities? Are there correlations with socioeconomic or lifestyle patterns in these cities?

Several factors influence repeat passenger rates in ride-hailing services across different cities. Key determinants include:

1. Quality of Service

Waiting Time: Shorter waiting times enhance passenger satisfaction, leading to higher repeat usage. Conversely, longer waiting times can deter passengers from reusing the service.


Service Reliability: Consistent and dependable service fosters trust and encourages repeat patronage.

2. Pricing Strategies

Competitive Pricing: Affordable fares attract cost-sensitive passengers, increasing the likelihood of repeat usage. However, unsustainable low fares can lead to driver dissatisfaction, potentially affecting service quality.


Dynamic Pricing Models: Surge pricing during peak hours may deter repeat usage among price-sensitive customers.

3. City Demographics and Socioeconomic Factors

Income Levels: Higher-income populations are more likely to adopt and frequently use ride-hailing services.


Vehicle Ownership: Lower rates of personal vehicle ownership correlate with increased reliance on ride-hailing services.

Age Distribution: Younger individuals, particularly millennials, show a higher propensity to use and reuse ride-hailing services.


4. Built Environment and Urban Design

Neighborhood Walkability: Areas with higher walkability indices may see reduced ride-hailing usage, as residents prefer walking over short-distance rides.


Public Transit Accessibility: Easy access to public transportation can decrease dependence on ride-hailing services, affecting repeat usage rates.

5. Socioeconomic and Lifestyle Patterns

Urban vs. Suburban Areas: Urban residents with active lifestyles and higher disposable incomes are more inclined to use ride-hailing services regularly.

Cultural Attitudes: Perceptions of convenience, safety, and social status associated with ride-hailing influence repeat usage.

In summary, repeat passenger rates in ride-hailing services are shaped by a complex interplay of service quality, pricing, demographic factors, urban design, and cultural attitudes. Understanding these correlations can help service providers tailor their offerings to enhance customer retention across diverse urban landscapes.

### 2. Tourism vs. Business Demand Impact

How do tourism seasons or local events (festivals, conferences) impact Goodcabs' demand patterns? Would tailoring marketing efforts to these events increase trip volume in tourism-oriented cities?

Analyzing Goodcabs' demand patterns during tourism seasons and local events in India's tier-2 cities can provide valuable insights into how these factors influence ride-hailing services. Here's a structured approach to understanding this impact:

1. Identifying Key Events and Tourism Seasons

India's tier-2 cities host numerous festivals, conferences, and tourism seasons that significantly affect local transportation demand. For instance:

Jaipur: The Jaipur Literature Festival, typically held in January, attracts a large number of visitors.

Kochi: The Kochi-Muziris Biennale, an international exhibition of contemporary art, occurs every two years from December to March.

Ahmedabad: The International Kite Festival in January draws enthusiasts from around the world.

These events, among others, lead to increased visitor inflow, thereby impacting demand for services like Goodcabs.

2. Analyzing Demand Patterns

To assess how these events influence Goodcabs' demand:

Data Segmentation: Compare trip volumes, fare amounts, and passenger counts during event periods against non-event periods.

Key Metrics:

Trip Volume: An increase during events indicates higher demand.
Average Fare: Fluctuations may reflect changes in trip distances or surge pricing.
Passenger Types: A rise in new passengers during events suggests tourist engagement.
3. Comparative Analysis

By evaluating these metrics, you can determine the extent to which events boost demand compared to regular days. For example, a significant uptick in trip volume during the Jaipur Literature Festival would highlight the event's impact on Goodcabs' operations in Jaipur.

4. Strategic Recommendations

Understanding these patterns enables Goodcabs to tailor marketing efforts effectively:

Targeted Promotions: Offer discounts or special services during major events to attract tourists.

Resource Allocation: Deploy additional drivers in cities during peak event times to meet increased demand.

Partnerships: Collaborate with event organizers for mutual promotions, enhancing visibility among attendees.

By aligning marketing strategies with local events and tourism seasons, Goodcabs can enhance trip volumes and establish a stronger presence in tourism-oriented tier-2 cities.

### 3. Emerging Mobility Trends and Goodcabs' Adaptation

What emerging mobility trends (such as electric vehicle adoption, green energy use) are impacting the cab service market in tier-2 cities? Should Goodcabs consider integrating electric vehicles or eco-friendly initiatives to stay competitive?

Emerging mobility trends, particularly the adoption of electric vehicles (EVs) and eco-friendly initiatives, are significantly impacting the cab service market in India's tier-2 cities.

## Electric Vehicle Adoption in Tier-2 Cities

Rising EV Sales: Tier-2 cities such as Surat, Jaipur, and Nagpur have witnessed substantial growth in electric two-wheeler sales, often surpassing figures in some tier-1 cities. For instance, in 2023, Surat sold 20,150 electric two-wheelers, while Jaipur sold 18,600, exceeding sales in larger cities like Ahmedabad and Mumbai.
BUSINESS STANDARD

Government Incentives: The Indian government has introduced schemes like PM E-DRIVE, allocating substantial funds to promote EV adoption across the country, including tier-2 cities. These initiatives aim to reduce pollution and encourage the use of cleaner fuels.
REUTERS

## Eco-Friendly Initiatives in the Cab Service Market

Transition to Electric Fleets: Cab service providers are increasingly adopting electric vehicles to offer sustainable transportation options. For example, BluSmart operates an all-electric cab fleet in regions like Delhi NCR and Bengaluru, setting a precedent for eco-friendly mobility solutions.
WIKIPEDIA

Collaborations for Green Mobility: Partnerships between companies, such as the collaboration between EaseMyTrip and BluSmart, are facilitating the introduction of environment-friendly cab services, highlighting a shift towards sustainable practices in the industry.
THE CS:RUNIVERSE

## Recommendations for Goodcabs

Integrate Electric Vehicles: Given the increasing acceptance and government support for EVs in tier-2 cities, incorporating electric vehicles into Goodcabs' fleet could enhance competitiveness and align with environmental sustainability goals.

Implement Eco-Friendly Initiatives: Adopting practices such as deploying electric cabs, utilizing renewable energy for operations, and offering incentives for eco-conscious customers can position Goodcabs as a leader in sustainable urban mobility.

By embracing these emerging trends, Goodcabs can not only stay competitive but also contribute to the broader goal of promoting clean and sustainable transportation solutions in India's growing urban centers.

## 4. Partnership Opportunities with Local Businesses

Are there opportunities for Goodcabs to partner with local businesses (such as hotels, malls, or event venues) to boost demand and improve customer loyalty?
Could these partnerships drive more traffic, especially in tourism-heavy or high-footfall areas?

Goodcabs has a significant opportunity to collaborate with local businesses in tier-2 cities to enhance its services and customer loyalty. Examples include:

Potential Partnerships
Hotels and Resorts: Partner with hotels to provide exclusive cab services for their guests, such as airport transfers or city tours. For example, partnering with luxury resorts during peak tourism seasons could guarantee a steady demand for premium cabs.

Shopping Malls: Establish designated pickup/drop-off points at high-footfall areas like malls. Offering discounts on rides for mall customers during events or sales could attract more passengers.

Event Venues: Collaborate with venues hosting weddings, conferences, or festivals to ensure reliable transportation for attendees.

Impact of Partnerships
Boost in Trip Volume: Targeted partnerships could lead to an increase in ride requests, especially during peak business hours or events.

Enhanced Customer Loyalty: Partnering with reputed businesses can position Goodcabs as a trusted and premium service provider, improving customer retention.

Actionable Strategies
Offer co-branded promotional offers (e.g., ride discounts for guests booking a stay at a partner hotel).
Implement a loyalty program that rewards passengers for frequent rides linked to partner businesses.

## 5. Data Collection for Enhanced Data-Driven Decisions

To make Goodcabs more data-driven and improve its performance across key metrics (such as repeat passenger rate, customer satisfaction, new passengers, and trip volume), what additional data should Goodcabs collect?
Consider data that could provide deeper insights into customer behavior, operational efficiency, and market trends.

Data Collection for Enhanced Data-Driven Decisions
To improve its performance and become more data-driven, Goodcabs should focus on collecting additional data points that provide deeper insights into customer behavior, operational efficiency, and market trends.

Additional Data to Collect
Customer Feedback Data:

Detailed passenger feedback on drivers, vehicle cleanliness, ride quality, and punctuality.
Suggestions for improvement to enhance customer satisfaction.
Trip-Specific Data:

Preferred pickup and drop-off locations to identify high-demand zones.
Average waiting time for passengers and response time for drivers.
Driver Performance Metrics:

Ratings given by passengers to drivers.
Frequency of on-time trips and cancellation rates by drivers.
Seasonal Trends:

Seasonal demand fluctuations, such as during festivals, public holidays, or school vacations.
Analysis of ride patterns (weekday vs. weekend rides).
Customer Demographics:

Passenger age group, profession, and preferences (e.g., eco-friendly vehicles or premium cabs).
Benefits of Collecting Additional Data
Improved Customer Retention: By understanding passenger preferences and addressing their concerns, Goodcabs can enhance the overall customer experience.

Operational Efficiency: Analyzing driver performance and high-demand areas can help optimize resource allocation.

Market Trends: Identifying patterns in customer behavior and seasonal trends enables more targeted marketing and promotions.

Next Steps for Goodcabs
Invest in technology to collect, analyze, and visualize data (e.g., dashboards for real-time insights).
Implement machine learning models to predict demand based on historical trends and external factors (e.g., weather, events).
Use insights to inform business decisions, such as expanding services to underserved areas or adding features to the app for a seamless booking experience.